In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import zipfile
import glob

# ⚠️ MANUAL UPDATE REQUIRED: update VDEM_VERSION when downloading a new version
# See docs/instructions_data_maintenance.md — VDEM section
VDEM_VERSION  = "16"
VDEM_FILENAME = f"vdem_full_v{VDEM_VERSION}.csv"
VDEM_PATH     = os.path.join(RAW_DIR, VDEM_FILENAME)

# Check file exists
if os.path.exists(VDEM_PATH):
    size_mb = os.path.getsize(VDEM_PATH) / (1024 * 1024)
    print(f"Found: {VDEM_FILENAME} ({size_mb:.1f} MB)")
else:
    print(f"FILE NOT FOUND: {VDEM_PATH}")
    print("See docs/instructions_data_maintenance.md — VDEM section")

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

FILE NOT FOUND: /Users/boulanger/Documents/governance-framework/data/raw/vdem_full_v16.csv
See docs/instructions_data_maintenance.md — VDEM section
Log loaded. Rows: 6
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


In [2]:
import json
from pathlib import Path

BOOTSTRAP = """import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os"""

notebooks_dir = Path(PROJECT_ROOT) / 'notebooks' / 'exploration'
notebooks = sorted(notebooks_dir.glob('*.ipynb'))

for nb_path in notebooks:
    with open(nb_path, 'r') as f:
        nb = json.load(f)
    
    first_cell = nb['cells'][0]
    if first_cell['cell_type'] == 'code':
        old_source = ''.join(first_cell['source'])
        if 'sys.path.insert' in old_source and 'Path.cwd()' not in old_source:
            # Replace just the sys.path.insert line, preserve rest of cell
            new_lines = []
            skip_next = False
            added_bootstrap = False
            for line in first_cell['source']:
                if 'import sys' in line and not added_bootstrap:
                    new_lines.append('import sys\n')
                    new_lines.append('from pathlib import Path\n')
                    new_lines.append('\n')
                    new_lines.append('# Bootstrap — find src/ dynamically, works on any machine\n')
                    new_lines.append("sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))\n")
                    new_lines.append('\n')
                    added_bootstrap = True
                elif 'sys.path.insert' in line:
                    continue  # remove old hardcoded line
                else:
                    new_lines.append(line)
            first_cell['source'] = new_lines
            with open(nb_path, 'w') as f:
                json.dump(nb, f, indent=1)
            print(f"Updated: {nb_path.name}")
        else:
            print(f"Skipped (already updated or no sys.path.insert): {nb_path.name}")

print("Done.")

Skipped (already updated or no sys.path.insert): 01_pdf_extraction.ipynb
Skipped (already updated or no sys.path.insert): 02_source_registry.ipynb
Skipped (already updated or no sys.path.insert): 03_vdem_pipeline.ipynb
Updated: 04_wgi_pipeline.ipynb
Updated: 05_wjp_pipeline.ipynb
Updated: 06_fh_fiw_pipeline.ipynb
Updated: 07_fsi_pipeline.ipynb
Updated: 08_ti_cpi_pipeline.ipynb
Done.


## V-Dem Pipeline

**Download instructions:** See `docs/instructions_data_maintenance.md` — VDEM section.

Once the file is in `data/raw/` and named `vdem_full_v{VERSION}.csv`, run the cells below.

In [ ]:
import zipfile
import glob

# Construct ZIP filename dynamically from VDEM_VERSION
zip_pattern = os.path.join(DOWNLOADS_DIR, f"V-Dem-CY-FullOthers-v{VDEM_VERSION}*.zip")
zip_matches = glob.glob(zip_pattern)

if not os.path.exists(VDEM_PATH):
    if zip_matches:
        with zipfile.ZipFile(zip_matches[0], 'r') as z:
            z.extract(f"V-Dem-CY-Full+Others-v{VDEM_VERSION}.csv", RAW_DIR)
        
        src = os.path.join(RAW_DIR, f"V-Dem-CY-Full+Others-v{VDEM_VERSION}.csv")
        if os.path.exists(VDEM_PATH):
            os.remove(VDEM_PATH)
        os.rename(src, VDEM_PATH)
        
        size_mb = os.path.getsize(VDEM_PATH) / (1024 * 1024)
        print(f"Extracted: {VDEM_FILENAME} ({size_mb:.1f} MB)")
    else:
        print(f"ZIP not found in Downloads — download v{VDEM_VERSION} from v-dem.net")
else:
    print(f"Already exists: {VDEM_FILENAME} ({os.path.getsize(VDEM_PATH)/(1024*1024):.1f} MB)")

In [ ]:
matches = glob.glob(os.path.join(DOWNLOADS_DIR, f"V-Dem-CY-FullOthers-v{VDEM_VERSION}*.zip"))

if not matches:
    print("ZIP not found")
else:
    zip_path = matches[0]
    print(f"Found: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        print("Contents:")
        for f in z.namelist():
            print(f"  {f}")

In [ ]:
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extract(f"V-Dem-CY-Full+Others-v{VDEM_VERSION}.csv", RAW_DIR)

# Rename to our standard convention
src = os.path.join(RAW_DIR, f"V-Dem-CY-Full+Others-v{VDEM_VERSION}.csv")
dst = os.path.join(RAW_DIR, VDEM_FILENAME)

# Confirm
size_mb = os.path.getsize(dst) / (1024 * 1024)
print(f"Extracted and renamed to: {VDEM_FILENAME}")
print(f"Size: {size_mb:.1f} MB")

In [ ]:
vdem = pd.read_csv(os.path.join(RAW_DIR, VDEM_FILENAME), low_memory=False)
print(f"Shape: {vdem.shape}")
print(f"Columns (first 20): {list(vdem.columns[:20])}")
print(f"Years: {vdem['year'].min()} — {vdem['year'].max()}")
print(f"Countries: {vdem['country_name'].nunique()}")

In [ ]:
# All V-Dem variables used in the framework
VDEM_VARS = [
    # Identifiers
    'country_name', 'country_text_id', 'country_id', 'year',

    # Political settlement
    'v2pepwrses', 'v2pepwrsoc', 'v2x_egal', 'v2psoppaut',

    # Political stability
    'v2x_regime',

    # State capacity
    'v2svstterr', 'v2svdomaut',

    # Government effectiveness
    'v2clrspct',

    # State control over economy
    'v2clstown',

    # Legislative checks
    'v2xlg_legcon', 'v2lgoppart', 'v2lgqstexp', 'v2lginvstp', 'v2lgotovst', 'v2x_horacc',

    # Judicial independence
    'v2juhcind', 'v2juncind', 'v2jucomp', 'v2jupack', 'v2jupurge',

    # Electoral process
    'v2x_polyarchy', 'v2elfrfair', 'v2elirreg', 'v2elintim', 'v2elvotbuy', 'v2elaccept',

    # Political participation
    'v2x_partip', 'v2psprlnks', 'v2pscohesv', 'v2cseeorgs', 'v2dlconslt', 'v2csreprss',

    # Civil liberties
    'v2x_civlib', 'v2x_clpriv', 'v2clrelig', 'v2cldmovem', 'v2cldmovew',
    'v2clsocgrp', 'v2clslavef',

    # Media freedom
    'v2x_freexp_altinf', 'v2mecenefm', 'v2meharjrn', 'v2mecorrpt', 'v2meslfcen',
    'v2merange', 'v2mebias', 'v2mecrit',

    # Civil society space
    'v2cscnsult', 'v2csprtcpt',

    # Government transparency
    'v2cltrnslw',

    # Legal quality and predictability
    'v2clacjstm', 'v2clacjstw', 'v2xeg_eqaccess',

    # Personal security
    'v2cltort', 'v2clkill', 'v2clrgunev',

    # Property rights
    'v2clprptym', 'v2clprptyw', 'v2xcl_prpty',

    # Corruption
    'v2x_corr', 'v2excrptps', 'v2exembez', 'v2lgcrrpt', 'v2jucorrdc',
]

# Check for any variables not found in dataset
missing = [v for v in VDEM_VARS if v not in vdem.columns]
if missing:
    print(f"Missing variables ({len(missing)}):")
    for v in missing:
        print(f"  {v}")
else:
    print("All variables found.")

In [ ]:
vdem_filtered = vdem[VDEM_VARS].copy()
vdem_filtered = vdem_filtered[vdem_filtered['year'] >=FRAMEWORK_START_YEAR]

print(f"Shape after filtering: {vdem_filtered.shape}")
print(f"Years: {vdem_filtered['year'].min()} — {vdem_filtered['year'].max()}")
print(f"Countries: {vdem_filtered['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (vdem_filtered.isnull().sum() / len(vdem_filtered) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

In [ ]:
output_path = os.path.join(PROCESSED_DIR, "vdem_filtered.csv")
vdem_filtered.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {vdem_filtered.shape}")

In [ ]:
# Remove original extracted filename if it exists
# Filename is constructed dynamically — no hardcoding needed
original = os.path.join(RAW_DIR, f"V-Dem-CY-Full+Others-v{VDEM_VERSION}.csv")
if os.path.exists(original):
    os.remove(original)
    print(f"Removed: {original}")
else:
    print("No original file to remove")

In [ ]:
import time
from datetime import datetime

# Derive as-of date from file modification date — no hardcoding needed
file_mod_time = os.path.getmtime(VDEM_PATH)
VDEM_AS_OF_DATE = datetime.fromtimestamp(file_mod_time).strftime("%Y-%m")

# Derive latest year directly from data
latest_year = str(int(vdem_filtered['year'].max()))

print(f"File last modified: {VDEM_AS_OF_DATE}")
print(f"Latest year in data: {latest_year}")

# Update download log — all values derived from data/metadata, no hardcoding
update_entry(
    "VDEM",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=VDEM_AS_OF_DATE,
    local_filename=VDEM_FILENAME,
    latest_available_version=f"v{VDEM_VERSION}",
    notes="Full+Others CSV. Extracted from ZIP. Manual download via email form required annually."
)

print_entry("VDEM")